# LeetCode 1845 — Seat Reservation Manager bằng Python

Link đề: https://leetcode.com/problems/seat-reservation-manager/ · Độ khó: **Medium** · Chủ đề: Design, Heap (Priority Queue)

Skeleton LeetCode:

```python
class SeatManager:

    def __init__(self, n: int):
        ...

    def reserve(self) -> int:
        ...

    def unreserve(self, seatNumber: int) -> None:
        ...


# Your SeatManager object will be instantiated and called as such:
# obj = SeatManager(n)
# param_1 = obj.reserve()
# obj.unreserve(seatNumber)
```

## 1. Mục tiêu học

1. Viết **bộ chạy chuỗi lệnh** kiểu LeetCode cho bài thiết kế lớp và sinh chuỗi lệnh ngẫu nhiên **hợp lệ** (đúng các cam kết của đề).
2. Hiểu thao tác cốt lõi: "lấy **số nhỏ nhất** trong tập ghế trống" + "trả một ghế về tập" ⇒ **min-heap**.
3. Tối ưu khởi tạo bằng **heap lười** (lazy): chỉ đưa vào heap những ghế đã từng bị đặt rồi trả lại, còn các ghế chưa từng dùng quản lý bằng một **con trỏ** `next_free`.
4. So sánh với mảng đánh dấu (quét tuyến tính), danh sách sắp xếp + `bisect.insort`, và **cây Fenwick** tìm vị trí trống đầu tiên.
5. Chứng minh invariant "mọi ghế trả lại đều nhỏ hơn `next_free`".

## 2. Đề bài & ví dụ

Thiết kế hệ thống quản lý `n` ghế đánh số `1..n`. Cài lớp `SeatManager`:
- `SeatManager(n)`: khởi tạo `n` ghế, **tất cả đều trống**.
- `reserve()`: lấy ghế trống có **số nhỏ nhất**, đặt chỗ và trả về số ghế đó.
- `unreserve(seatNumber)`: hủy đặt ghế `seatNumber` (ghế trở lại trống).

**Ví dụ 1:**
```
["SeatManager","reserve","reserve","unreserve","reserve","reserve","reserve","reserve","unreserve"]
[[5],[],[],[2],[],[],[],[],[5]]
Output: [null,1,2,null,2,3,4,5,null]
```

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ seatNumber ≤ n`; mỗi lần `reserve` **đảm bảo** còn ít nhất một ghế trống; mỗi lần `unreserve` **đảm bảo** ghế đó đang được đặt; tổng số lời gọi `≤ 10⁵`.

In [1]:
import random
import time
import heapq
import bisect

def run(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["SeatManager", "reserve", "reserve", "unreserve", "reserve", "reserve", "reserve", "reserve", "unreserve"]
ex_args = [[5], [], [], [2], [], [], [], [], [5]]
ex_out = [None, 1, 2, None, 2, 3, 4, 5, None]
print(list(zip(ex_ops, ex_args)), "->", ex_out)

[('SeatManager', [5]), ('reserve', []), ('reserve', []), ('unreserve', [2]), ('reserve', []), ('reserve', []), ('reserve', []), ('reserve', []), ('unreserve', [5])] -> [None, 1, 2, None, 2, 3, 4, 5, None]


## 3. Kiến thức nền

### 3.1 Bài thiết kế và bộ chạy lệnh

LeetCode cho hai mảng: tên phương thức và tham số. Lệnh đầu tạo đối tượng (kết quả `null` ↔ `None`), các lệnh sau gọi phương thức. Hàm `run` ở trên mô phỏng đúng quy trình này.

### 3.2 Min-heap `heapq`

`heapify(list)` `O(n)`; `heappop` lấy nhỏ nhất `O(log n)`; `heappush` `O(log n)`. Danh sách `[1, 2, …, n]` **đã sắp** nên tự nó đã là một heap hợp lệ (không cần `heapify`).

In [2]:
h = list(range(1, 6))
print("list sắp sẵn là heap?", all(h[i] <= h[c] for i in range(len(h)) for c in (2 * i + 1, 2 * i + 2) if c < len(h)))
print(heapq.heappop(h), heapq.heappop(h)); heapq.heappush(h, 2); print("sau khi trả 2:", heapq.heappop(h))

list sắp sẵn là heap? True
1 2
sau khi trả 2: 2


### 3.3 `bisect.insort` — chèn vào list đã sắp

Giữ list đã sắp: `insort(a, x)` tìm vị trí `O(log n)` nhưng chèn tốn `O(n)` (dịch phần tử) — thường vẫn nhanh nhờ `memmove` trong C.

In [3]:
a = [3, 4, 5]; bisect.insort(a, 1); bisect.insort(a, 4)
print(a, "| lấy nhỏ nhất:", a.pop(0), a)

[3, 4, 4, 5] | lấy nhỏ nhất: 1 [3, 4, 4, 5]


### 3.4 Mảng đánh dấu

`taken[i] = True` nếu ghế `i` đã đặt. Tìm ghế trống nhỏ nhất = quét từ 1 — `O(n)` mỗi lần.

### 3.5 Cây Fenwick: "vị trí trống đầu tiên"

Lưu `free[i] ∈ {0,1}` trong BIT; ghế trống nhỏ nhất là vị trí có tổng tiền tố bằng 1 lần đầu — tìm bằng **nhảy nhị phân** `O(log n)` (xem lại kỹ thuật ở bài 1756).

## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay Ví dụ 1 với **tập ghế trống**:

| lệnh | tập trống trước | trả về | tập trống sau |
|---|---|---|---|
| reserve | {1,2,3,4,5} | 1 | {2,3,4,5} |
| reserve | {2,3,4,5} | 2 | {3,4,5} |
| unreserve(2) | {3,4,5} | — | {2,3,4,5} |
| reserve | {2,3,4,5} | 2 | {3,4,5} |
| reserve ×3 | … | 3, 4, 5 | {} |
| unreserve(5) | {} | — | {5} |

Hai thao tác trên tập: **lấy min** và **thêm phần tử** ⇒ min-heap.

**Quan sát thêm:** các ghế được đặt lần đầu theo đúng thứ tự `1, 2, 3, …`. Một ghế được trả lại luôn có số **nhỏ hơn** mọi ghế chưa từng được đặt. Vậy chỉ cần heap cho các ghế **đã trả lại**, còn "ghế mới" là một con trỏ tăng dần.

## 5. Cách 1 — Brute Force / baseline: mảng đánh dấu + quét

In [4]:
class SeatBrute:
    def __init__(self, n):
        self.taken = [False] * (n + 1)
    def reserve(self):
        i = 1
        while self.taken[i]:
            i += 1
        self.taken[i] = True
        return i
    def unreserve(self, seatNumber):
        self.taken[seatNumber] = False

got = run(SeatBrute, ex_ops, ex_args); print(got); assert got == ex_out

[None, 1, 2, None, 2, 3, 4, 5, None]


**Độ phức tạp:** `reserve` `O(n)` tệ nhất ⇒ `10⁵` lệnh × `10⁵` = `10¹⁰` khi các ghế đầu luôn bị chiếm.

**Nút thắt:** mỗi lần `reserve` quét lại các ghế đã đặt ở đầu dãy. Cần cấu trúc "trả min" nhanh.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (min-heap):** giữ tập ghế trống trong min-heap: `reserve = heappop`, `unreserve = heappush`.

**Insight 2 (heap lười + con trỏ):** gọi `next_free` = ghế nhỏ nhất **chưa từng** được đặt (ban đầu 1). Heap `returned` chỉ chứa ghế đã từng đặt rồi trả lại.
- `reserve`: **nếu** `returned` không rỗng **thì** `heappop(returned)`; **ngược lại** trả `next_free` và tăng nó.
- `unreserve(s)`: `heappush(returned, s)`.

**Invariant:** mọi phần tử của `returned` `< next_free` (vì một ghế chỉ được trả lại sau khi đã được đặt, và mọi ghế đã đặt đều `< next_free`). Do đó khi `returned` không rỗng, min của nó là ghế trống nhỏ nhất toàn cục.

Khởi tạo chỉ `O(1)` thay vì `O(n)`, heap nhỏ hơn nhiều.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Min-heap chứa mọi ghế trống

In [5]:
class SeatHeap:
    def __init__(self, n):
        self.h = list(range(1, n + 1))        # đã sắp => là heap hợp lệ
    def reserve(self):
        return heapq.heappop(self.h)
    def unreserve(self, seatNumber):
        heapq.heappush(self.h, seatNumber)

print(run(SeatHeap, ex_ops, ex_args))

[None, 1, 2, None, 2, 3, 4, 5, None]


**Code nộp LeetCode (Cách 2):** đổi tên lớp thành `SeatManager`. **Độ phức tạp:** khởi tạo `O(n)`, mỗi lệnh `O(log n)`, Space `O(n)`.

### 7.2 Cách 3 — Heap lười + con trỏ `next_free`

In [6]:
class SeatLazy:
    def __init__(self, n):
        self.next_free = 1
        self.returned = []
    def reserve(self):
        if self.returned:
            return heapq.heappop(self.returned)
        seat = self.next_free
        self.next_free += 1
        return seat
    def unreserve(self, seatNumber):
        heapq.heappush(self.returned, seatNumber)

print(run(SeatLazy, ex_ops, ex_args))

[None, 1, 2, None, 2, 3, 4, 5, None]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** khởi tạo `O(1)`, mỗi lệnh `O(log r)` với `r` = số ghế đang nằm trong `returned` (`≤` số lệnh), Space `O(r)`.

### 7.3 Cách 4 — List đã sắp + `bisect.insort`

In [7]:
class SeatSortedList:
    def __init__(self, n):
        self.next_free = 1
        self.returned = []                     # giữ sắp tăng
    def reserve(self):
        if self.returned:
            return self.returned.pop(0)        # O(r) dịch phần tử
        seat = self.next_free; self.next_free += 1
        return seat
    def unreserve(self, seatNumber):
        bisect.insort(self.returned, seatNumber)

print(run(SeatSortedList, ex_ops, ex_args))

[None, 1, 2, None, 2, 3, 4, 5, None]


**Độ phức tạp Cách 4:** mỗi lệnh `O(r)` tệ nhất (dịch mảng) nhưng hằng số rất nhỏ. Có thể giữ list **giảm dần** để `pop()` ở cuối `O(1)` — xem Bài tập 4.

### 7.4 Cách 5 — Cây Fenwick + nhảy nhị phân (tìm ghế trống đầu tiên)

Mảng `free[i] = 1` nếu ghế `i` trống; BIT trên `free`. Ghế trống nhỏ nhất = vị trí nhỏ nhất có tổng tiền tố `≥ 1`. Cách này còn trả lời được "ghế trống thứ `k`" — hữu ích cho các biến thể.

In [8]:
class SeatFenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
        for i in range(1, n + 1):              # dựng BIT với mọi free[i] = 1 trong O(n)
            self.t[i] += 1
            j = i + (i & -i)
            if j <= n: self.t[j] += self.t[i]
        self.LOG = 1 << n.bit_length()
    def _add(self, i, v):
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def _first_free(self):
        pos, k, step = 0, 1, self.LOG
        while step:
            nxt = pos + step
            if nxt <= self.n and self.t[nxt] < k:
                pos = nxt; k -= self.t[nxt]
            step >>= 1
        return pos + 1
    def reserve(self):
        s = self._first_free(); self._add(s, -1); return s
    def unreserve(self, seatNumber):
        self._add(seatNumber, 1)

print(run(SeatFenwick, ex_ops, ex_args))

[None, 1, 2, None, 2, 3, 4, 5, None]


**Độ phức tạp Cách 5:** khởi tạo `O(n)`, mỗi lệnh `O(log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 — in `next_free` và heap `returned` sau mỗi lệnh:

| lệnh | trả về | next_free | returned |
|---|---|---|---|
| SeatManager(5) | — | 1 | [] |
| reserve | 1 | 2 | [] |
| reserve | 2 | 3 | [] |
| unreserve(2) | — | 3 | [2] |
| reserve | 2 (từ heap) | 3 | [] |
| reserve | 3 | 4 | [] |
| reserve | 4 | 5 | [] |
| reserve | 5 | 6 | [] |
| unreserve(5) | — | 6 | [5] |

In [9]:
class TraceLazy(SeatLazy):
    def reserve(self):
        src = "heap" if self.returned else "con trỏ"
        s = super().reserve(); print(f"  reserve()      -> {s} (từ {src}) | next_free={self.next_free}, returned={sorted(self.returned)}"); return s
    def unreserve(self, seatNumber):
        super().unreserve(seatNumber); print(f"  unreserve({seatNumber})   | next_free={self.next_free}, returned={sorted(self.returned)}")

assert run(TraceLazy, ex_ops, ex_args) == ex_out

  reserve()      -> 1 (từ con trỏ) | next_free=2, returned=[]
  reserve()      -> 2 (từ con trỏ) | next_free=3, returned=[]
  unreserve(2)   | next_free=3, returned=[2]
  reserve()      -> 2 (từ heap) | next_free=3, returned=[]
  reserve()      -> 3 (từ con trỏ) | next_free=4, returned=[]
  reserve()      -> 4 (từ con trỏ) | next_free=5, returned=[]
  reserve()      -> 5 (từ con trỏ) | next_free=6, returned=[]
  unreserve(5)   | next_free=6, returned=[5]


Trace một chuỗi tự dựng có nhiều ghế trả lại không theo thứ tự (ví dụ tự dựng — đối chiếu brute force; nên đối chiếu lại đề gốc):

In [10]:
ops = ["SeatManager"] + ["reserve"] * 5 + ["unreserve", "unreserve", "unreserve", "reserve", "reserve", "reserve", "reserve"]
args = [[6]] + [[]] * 5 + [[4], [1], [3], [], [], [], []]
r = run(TraceLazy, ops, args); print(r); assert r == run(SeatBrute, ops, args)

  reserve()      -> 1 (từ con trỏ) | next_free=2, returned=[]
  reserve()      -> 2 (từ con trỏ) | next_free=3, returned=[]
  reserve()      -> 3 (từ con trỏ) | next_free=4, returned=[]
  reserve()      -> 4 (từ con trỏ) | next_free=5, returned=[]
  reserve()      -> 5 (từ con trỏ) | next_free=6, returned=[]
  unreserve(4)   | next_free=6, returned=[4]
  unreserve(1)   | next_free=6, returned=[1, 4]
  unreserve(3)   | next_free=6, returned=[1, 3, 4]
  reserve()      -> 1 (từ heap) | next_free=6, returned=[3, 4]
  reserve()      -> 3 (từ heap) | next_free=6, returned=[4]
  reserve()      -> 4 (từ heap) | next_free=6, returned=[]
  reserve()      -> 6 (từ con trỏ) | next_free=7, returned=[]
[None, 1, 2, 3, 4, 5, None, None, None, 1, 3, 4, 6]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — heap lười + con trỏ.

In [11]:
class SeatManager:

    def __init__(self, n: int):
        self.next_free = 1
        self.returned = []

    def reserve(self) -> int:
        if self.returned:
            return heapq.heappop(self.returned)
        seat = self.next_free
        self.next_free += 1
        return seat

    def unreserve(self, seatNumber: int) -> None:
        heapq.heappush(self.returned, seatNumber)


# Your SeatManager object will be instantiated and called as such:
# obj = SeatManager(n)
# param_1 = obj.reserve()
# obj.unreserve(seatNumber)
got = run(SeatManager, ex_ops, ex_args); print(got); assert got == ex_out

[None, 1, 2, None, 2, 3, 4, 5, None]


## 10. Vì sao thuật toán đúng

Gọi `Free` = tập ghế trống thật sự. **Invariant:** (I1) `Free = set(returned) ∪ {next_free, next_free+1, …, n}`; (I2) mọi phần tử của `returned` `< next_free`; (I3) `returned` không có phần tử trùng.

- **Khởi tạo:** `returned = []`, `next_free = 1` ⇒ `Free = {1..n}` ✓.
- **`reserve`:** nếu `returned` khác rỗng, theo (I2) `min(returned) < next_free ≤` mọi ghế "mới" ⇒ `min(returned) = min(Free)` ✓; pop giữ (I1)–(I3). Nếu rỗng, `min(Free) = next_free` (đề đảm bảo còn ghế trống nên `next_free ≤ n`) ⇒ trả nó và tăng con trỏ ✓.
- **`unreserve(s)`:** đề đảm bảo `s` đang được đặt ⇒ `s ∉ Free`; ghế đã đặt thì hoặc `< next_free` (đã từng cấp từ con trỏ) ⇒ (I2) giữ; `s` chưa có trong `returned` ⇒ (I3) giữ; thêm vào ⇒ (I1) ✓. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n` = số ghế, `Q` = số lời gọi (`≤ 10⁵`), `r` = số ghế đang ở `returned`.

| Cách | `__init__` | `reserve` | `unreserve` | Space |
|---|---|---|---|---|
| 1. Mảng đánh dấu | `O(n)` | `O(n)` | `O(1)` | `O(n)` |
| 2. Heap đầy đủ | `O(n)` | `O(log n)` | `O(log n)` | `O(n)` |
| 3. Heap lười | `O(1)` | `O(log r)` | `O(log r)` | `O(r)` |
| 4. List sắp + insort | `O(1)` | `O(r)` | `O(r)` | `O(r)` |
| 5. Fenwick | `O(n)` | `O(log n)` | `O(log n)` | `O(n)` |

In [12]:
def random_ops(rng, n, q):
    """Sinh chuỗi lệnh HỢP LỆ: reserve khi còn ghế trống, unreserve một ghế đang được đặt (chọn ngẫu nhiên O(1))."""
    ops, args = ["SeatManager"], [[n]]
    reserved, where = [], {}                       # danh sách ghế đang đặt + vị trí trong danh sách
    free_returned, next_new = [], 1                # mô phỏng ghế nhỏ nhất để biết ghế nào được đặt
    for _ in range(q):
        if reserved and (len(reserved) == n or rng.random() < 0.45):
            j = rng.randrange(len(reserved)); s = reserved[j]
            last = reserved.pop(); 
            if last != s:
                reserved[j] = last; where[last] = j
            del where[s]
            ops.append("unreserve"); args.append([s]); heapq.heappush(free_returned, s)
        else:
            s = heapq.heappop(free_returned) if free_returned else next_new
            if s == next_new: next_new += 1
            where[s] = len(reserved); reserved.append(s)
            ops.append("reserve"); args.append([])
    return ops, args

rng = random.Random(1845)
ops, args = random_ops(rng, 10 ** 5, 10 ** 5)
for cls in (SeatHeap, SeatManager, SeatSortedList, SeatFenwick):
    t0 = time.perf_counter(); run(cls, ops, args); print(f"{cls.__name__:<15} {(time.perf_counter()-t0)*1000:7.1f} ms")
ops2 = ["SeatManager"] + ["reserve"] * 3000 + ["unreserve", "reserve"] * 3000
args2 = [[5000]] + [[]] * 3000 + [[1], []] * 3000
t0 = time.perf_counter(); run(SeatBrute, ops2, args2); print(f"SeatBrute (tệ nhất 3000 ghế đầu bị chiếm): {(time.perf_counter()-t0)*1000:.0f} ms")

SeatHeap           28.2 ms
SeatManager        13.0 ms
SeatSortedList     12.0 ms


SeatFenwick       191.7 ms
SeatBrute (tệ nhất 3000 ghế đầu bị chiếm): 84 ms


## 12. Bộ test

Sinh chuỗi lệnh ngẫu nhiên **hợp lệ** (seed cố định): chỉ `reserve` khi còn ghế trống, chỉ `unreserve` ghế đang được đặt. Đối chiếu mọi cài đặt với `SeatBrute`.

In [13]:
IMPLS = [SeatHeap, SeatLazy, SeatSortedList, SeatFenwick, SeatManager]
def check(ops, args):
    ref = run(SeatBrute, ops, args)
    for cls in IMPLS:
        assert run(cls, ops, args) == ref, (cls.__name__, ops, args)
    return ref

assert check(ex_ops, ex_args) == ex_out
edge = [
    (["SeatManager", "reserve", "unreserve", "reserve"], [[1], [], [1], []], [None, 1, None, 1]),
    (["SeatManager", "reserve", "reserve", "reserve", "unreserve", "unreserve", "reserve", "reserve"], [[3], [], [], [], [3], [1], [], []],
     [None, 1, 2, 3, None, None, 1, 3]),
    (["SeatManager", "reserve", "reserve", "unreserve", "unreserve", "reserve", "reserve", "reserve"], [[4], [], [], [2], [1], [], [], []],
     [None, 1, 2, None, None, 1, 2, 3]),
]
for ops, args, exp in edge:
    assert check(ops, args) == exp
rng = random.Random(18450)
for _ in range(500):
    check(*random_ops(rng, rng.randint(1, 10), rng.randint(1, 30)))
for _ in range(5):
    check(*random_ops(rng, rng.randint(50, 500), 3000))
print("Ví dụ, edge case, 500 chuỗi ngắn, 5 chuỗi 3000 lệnh: OK")

Ví dụ, edge case, 500 chuỗi ngắn, 5 chuỗi 3000 lệnh: OK


### 12.1 Kiểm tra tính chất trên trạng thái

Song song với đối tượng `SeatManager`, giữ tập ghế đang đặt `taken`. Sau mỗi `reserve` kiểm tra: (1) ghế trả về chưa bị đặt; (2) nó là ghế trống **nhỏ nhất**; (3) không vượt `n`.

In [14]:
rng = random.Random(31)
for _ in range(200):
    n = rng.randint(1, 15); ops, args = random_ops(rng, n, 40)
    obj = SeatManager(n); taken = set()
    for op, a in zip(ops[1:], args[1:]):
        if op == "reserve":
            s_ = obj.reserve()
            assert s_ not in taken and 1 <= s_ <= n and s_ == min(set(range(1, n + 1)) - taken)
            taken.add(s_)
        else:
            obj.unreserve(*a); taken.discard(a[0])
print("Tính chất 'luôn trả ghế trống nhỏ nhất, không cấp trùng' đúng trên 200 chuỗi lệnh")

Tính chất 'luôn trả ghế trống nhỏ nhất, không cấp trùng' đúng trên 200 chuỗi lệnh


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng **stack** (LIFO) cho ghế trả lại

In [15]:
class WrongStack(SeatLazy):
    def reserve(self):
        if self.returned:
            return self.returned.pop()                   # SAI: lấy ghế trả lại GẦN NHẤT, không phải nhỏ nhất
        s = self.next_free; self.next_free += 1; return s
    def unreserve(self, seatNumber):
        self.returned.append(seatNumber)

ops, args, exp = edge[1]
print("sai :", run(WrongStack, ops, args), "| đúng:", exp)

sai : [None, 1, 2, 3, None, None, 1, 3] | đúng: [None, 1, 2, 3, None, None, 1, 3]


**Vì sao sai:** ở chuỗi lệnh trên, stack **tình cờ** cho đúng vì ghế trả sau cùng (1) cũng là ghế nhỏ nhất. Chỉ cần đổi thứ tự trả (trả 1 trước, 3 sau) là lộ lỗi — stack trả ghế 3 trong khi ghế 1 đang trống:

In [16]:
ops = ["SeatManager", "reserve", "reserve", "reserve", "unreserve", "unreserve", "reserve"]
args = [[3], [], [], [], [1], [3], []]                    # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongStack, ops, args), "| đúng:", run(SeatManager, ops, args))

sai : [None, 1, 2, 3, None, None, 3] | đúng: [None, 1, 2, 3, None, None, 1]


**Sửa:** min-heap.

### Lỗi 2 — Heap lười nhưng ưu tiên con trỏ trước heap

In [17]:
class WrongPointerFirst(SeatLazy):
    def __init__(self, n):
        super().__init__(n); self.n = n
    def reserve(self):
        if self.next_free <= self.n:                     # SAI: ghế mới luôn lớn hơn ghế trả lại
            s = self.next_free; self.next_free += 1; return s
        return heapq.heappop(self.returned)

print("sai :", run(WrongPointerFirst, ex_ops, ex_args), "| đúng:", ex_out)

sai : [None, 1, 2, None, 3, 4, 5, 2, None] | đúng: [None, 1, 2, None, 2, 3, 4, 5, None]


**Vì sao sai:** ghế trả lại (2) nhỏ hơn mọi ghế mới (`≥ 3`) — phải lấy từ heap trước. **Sửa:** kiểm tra `returned` trước.

### Lỗi 3 — Heap đầy đủ nhưng `unreserve` không đẩy lại (quên trả ghế)

In [18]:
class WrongForget(SeatHeap):
    def unreserve(self, seatNumber):
        pass                                             # SAI

print("sai :", run(WrongForget, ex_ops[:5], ex_args[:5]), "| đúng:", ex_out[:5])

sai : [None, 1, 2, None, 3] | đúng: [None, 1, 2, None, 2]


**Vì sao sai:** ghế 2 không bao giờ trống lại ⇒ `reserve` trả 3 thay vì 2. **Sửa:** `heappush`.

### Lỗi 4 — Dùng `set` và `min(set)` mỗi lần (đúng nhưng `O(n)`)

In [19]:
class SlowSetMin:
    def __init__(self, n): self.free = set(range(1, n + 1))
    def reserve(self):
        s = min(self.free); self.free.remove(s); return s
    def unreserve(self, seatNumber): self.free.add(seatNumber)

ops = ["SeatManager"] + ["reserve"] * 3000; args = [[20000]] + [[]] * 3000
for cls in (SlowSetMin, SeatManager):
    t0 = time.perf_counter(); run(cls, ops, args); print(f"{cls.__name__:<11} {(time.perf_counter()-t0)*1000:7.1f} ms")

SlowSetMin    674.4 ms
SeatManager     0.3 ms


**Vì sao không tốt:** `min(set)` duyệt cả tập ⇒ `O(n)` mỗi lệnh. **Sửa:** heap (`O(log n)`).

### Lỗi 5 — Sinh test không tôn trọng cam kết của đề

Nếu bộ test gọi `unreserve` một ghế đang trống, heap lười sẽ chứa **trùng lặp** và trả cùng một ghế hai lần — nhưng đó là input **không hợp lệ** theo đề. Khi viết test cho bài thiết kế, luôn sinh lệnh đúng điều kiện ràng buộc (như hàm `random_ops`).

In [20]:
ops = ["SeatManager", "unreserve", "reserve", "reserve"]; args = [[2], [1], [], []]   # unreserve ghế chưa đặt: KHÔNG hợp lệ
print("input không hợp lệ -> heap lười:", run(SeatManager, ops, args), "| mảng đánh dấu:", run(SeatBrute, ops, args))

input không hợp lệ -> heap lười: [None, None, 1, 1] | mảng đánh dấu: [None, None, 1, 2]


Hai cài đặt cho kết quả khác nhau trên input **sai luật** — không phải lỗi của lời giải, mà là lỗi của test.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lấy phần tử **nhỏ nhất** trong một tập thay đổi" ⇒ **min-heap**.
- Tập ban đầu là dãy liên tiếp `1..n` ⇒ **con trỏ + heap lười** để khởi tạo `O(1)`.
- Bài thiết kế ⇒ bộ chạy lệnh + bản vét cạn + sinh lệnh hợp lệ.
- Cần "phần tử thứ k" hoặc "nhỏ nhất ≥ x" ⇒ Fenwick / cây phân đoạn / `SortedList`.

## 15. Biến thể của bài

1. **Trả về ghế trống nhỏ nhất `≥ x`** (đặt chỗ từ hàng x trở đi): Fenwick hoặc cây phân đoạn (Bài tập 5).
2. **2336 Smallest Number in Infinite Set**: gần giống hệt (tập vô hạn + addBack) — đúng kỹ thuật heap lười.
3. **855 Exam Room**: chọn ghế xa người khác nhất — cấu trúc khoảng trống.
4. **1146 / 1845 dạng bộ cấp phát ID** (allocator): cấp ID nhỏ nhất chưa dùng.
5. **Đặt nhiều ghế liên tiếp** (k ghế cạnh nhau): cây phân đoạn quản lý đoạn trống dài nhất.

Minh họa biến thể 2 (**2336 Smallest Number in Infinite Set**): tập vô hạn `{1, 2, 3, …}`, `popSmallest()` và `addBack(x)` (chỉ thêm nếu `x` đang không có trong tập). Khác bài gốc: `addBack` có thể gọi với số **đang có sẵn** ⇒ cần thêm một `set` để tránh trùng.

In [21]:
class SmallestInfiniteSet:
    def __init__(self):
        self.next_free = 1; self.heap = []; self.inheap = set()
    def popSmallest(self):
        if self.heap:
            x = heapq.heappop(self.heap); self.inheap.discard(x); return x
        self.next_free += 1; return self.next_free - 1
    def addBack(self, num):
        if num < self.next_free and num not in self.inheap:
            heapq.heappush(self.heap, num); self.inheap.add(num)

print(run(SmallestInfiniteSet, ["SmallestInfiniteSet", "addBack", "popSmallest", "popSmallest", "popSmallest", "addBack", "popSmallest", "popSmallest", "popSmallest"],
          [[], [2], [], [], [], [1], [], [], []]))

[None, None, 1, 2, 3, None, 1, 4, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | reserve | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng đánh dấu | Quét từ 1 | `O(n)` | `O(n)` | Đối chiếu |
| 2. Heap đầy đủ | Heap mọi ghế trống | `O(log n)` | `O(n)` | Đơn giản, đủ qua |
| 3. Heap lười | Con trỏ + heap ghế trả lại | `O(log r)` | `O(r)` | **Bản nộp** |
| 4. List + insort | Giữ list sắp | `O(r)` | `O(r)` | Hằng số nhỏ, `r` nhỏ |
| 5. Fenwick | Tìm vị trí trống đầu | `O(log n)` | `O(n)` | Biến thể "thứ k", "≥ x" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `SeatManager(3)`, `reserve, reserve, unreserve(1), reserve, reserve, unreserve(2), unreserve(3), reserve`.

**Bài 2 (tính tay):** Với Cách 3, ghi `next_free` và `returned` sau mỗi lệnh của Bài 1.

**Bài 3 (điền TODO):** Hoàn thiện `reserve_todo(next_free, returned)` trả `(ghế, next_free_mới)`.

**Bài 4 (tự viết lại):** Viết `SeatDescList` dùng list sắp **giảm dần** để `reserve` là `pop()` `O(1)` (chèn bằng `bisect` trên khóa âm).

**Bài 5 (mini challenge):** Viết `SeatFromRow` với thêm phương thức `reserve_at_least(x)` trả ghế trống nhỏ nhất `≥ x` (hoặc `−1`), dùng Fenwick.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reserve_todo(next_free, returned):
    # TODO: nếu returned có phần tử thì heappop; ngược lại dùng next_free và tăng nó
    return None, next_free

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class SeatDescList:
    def __init__(self, n):
        # TODO: next_free, list giảm dần
        pass

class SeatFromRow:
    def __init__(self, n):
        # TODO: Fenwick free[i]; reserve_at_least(x): k = prefix(x-1) + 1 -> tìm vị trí trống thứ k
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: mô phỏng tập trống. Mức 2: sau `reserve ×2`, `unreserve(1)` ⇒ trống {1, 3}. Mức 3: kết quả `[None, 1, 2, None, 1, 3, None, None, 2]`.

**Bài 2.** Mức 1: `next_free` lên 3 sau hai lần đặt. Mức 2: `unreserve(1)` ⇒ `returned = [1]`; `reserve` lấy 1; `reserve` lấy 3 (next_free = 4). Mức 3: `unreserve(2)`, `unreserve(3)` ⇒ `returned = [2, 3]` ⇒ `reserve` = 2.

**Bài 3.** Mức 1: `if returned`. Mức 2: `heappop`. Mức 3: trả tuple.

**Bài 4.** Mức 1: muốn `pop()` ở cuối trả ghế nhỏ nhất ⇒ list phải **giảm dần**. Mức 2: giữ list `neg` chứa `−ghế` sắp **tăng dần** (tương đương ghế giảm dần) ⇒ chèn bằng `insort(neg, −x)`. Mức 3: ghế nhỏ nhất là `−neg[−1]` ⇒ `reserve` = `−neg.pop()`; đối chiếu bằng test ngẫu nhiên.

**Bài 5.** Mức 1: `prefix(x − 1)` = số ghế trống `< x`. Mức 2: ghế trống nhỏ nhất `≥ x` là ghế trống thứ `prefix(x−1) + 1`. Mức 3: nếu thứ hạng đó vượt tổng số ghế trống ⇒ `−1`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
ops1 = ["SeatManager", "reserve", "reserve", "unreserve", "reserve", "reserve", "unreserve", "unreserve", "reserve"]
args1 = [[3], [], [], [1], [], [], [2], [3], []]
b1 = check(ops1, args1); print("Bài 1:", b1); assert b1 == [None, 1, 2, None, 1, 3, None, None, 2]
print("Bài 2:"); run(TraceLazy, ops1, args1)
def reserve_todo(next_free, returned):
    if returned:
        return heapq.heappop(returned), next_free
    return next_free, next_free + 1
r = []; assert reserve_todo(1, r) == (1, 2); r = [3, 5]; assert reserve_todo(7, r) == (3, 7)
print("Bài 3: OK")

Bài 1: [None, 1, 2, None, 1, 3, None, None, 2]
Bài 2:
  reserve()      -> 1 (từ con trỏ) | next_free=2, returned=[]
  reserve()      -> 2 (từ con trỏ) | next_free=3, returned=[]
  unreserve(1)   | next_free=3, returned=[1]
  reserve()      -> 1 (từ heap) | next_free=3, returned=[]
  reserve()      -> 3 (từ con trỏ) | next_free=4, returned=[]
  unreserve(2)   | next_free=4, returned=[2]
  unreserve(3)   | next_free=4, returned=[2, 3]
  reserve()      -> 2 (từ heap) | next_free=4, returned=[3]
Bài 3: OK


In [25]:
# Bài 4, 5
class SeatDescList:
    def __init__(self, n):
        self.next_free = 1; self.neg = []              # neg chứa -ghế, tăng dần => ghế giảm dần, ghế nhỏ nhất ở CUỐI
    def reserve(self):
        if self.neg:
            return -self.neg.pop()
        s = self.next_free; self.next_free += 1; return s
    def unreserve(self, seatNumber):
        bisect.insort(self.neg, -seatNumber)

class SeatFromRow(SeatFenwick):
    def _prefix(self, i):
        s = 0
        while i > 0: s += self.t[i]; i -= i & -i
        return s
    def _kth_free(self, k):
        pos, step = 0, self.LOG
        while step:
            nxt = pos + step
            if nxt <= self.n and self.t[nxt] < k:
                pos = nxt; k -= self.t[nxt]
            step >>= 1
        return pos + 1
    def reserve_at_least(self, x):
        k = self._prefix(x - 1) + 1
        if k > self._prefix(self.n):
            return -1
        s = self._kth_free(k); self._add(s, -1); return s

class BruteFromRow(SeatBrute):
    def __init__(self, n): super().__init__(n); self.n = n
    def reserve_at_least(self, x):
        for i in range(x, self.n + 1):
            if not self.taken[i]:
                self.taken[i] = True; return i
        return -1

rng = random.Random(7)
for _ in range(300):
    ops, args = random_ops(rng, rng.randint(1, 10), rng.randint(1, 25))
    assert run(SeatDescList, ops, args) == run(SeatBrute, ops, args)
    n = rng.randint(1, 10); ops5, args5, taken = ["SeatManager"], [[n]], set()
    for _ in range(25):
        if taken and rng.random() < 0.4:
            s = rng.choice(sorted(taken)); taken.discard(s); ops5.append("unreserve"); args5.append([s])
        else:
            x = rng.randint(1, n); ops5.append("reserve_at_least"); args5.append([x])
            free_ge = [i for i in range(x, n + 1) if i not in taken]
            if free_ge: taken.add(free_ge[0])
    assert run(SeatFromRow, ops5, args5) == run(BruteFromRow, ops5, args5)
print("Bài 4, 5: OK |", run(SeatFromRow, ["SeatManager", "reserve_at_least", "reserve_at_least", "reserve_at_least"], [[3], [2], [2], [2]]))

Bài 4, 5: OK | [None, 2, 3, -1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "con trỏ + heap lười":**

```python
next_free, returned = 1, []
def reserve():
    if returned: return heappop(returned)       # ghế trả lại luôn < next_free
    s = next_free; next_free += 1; return s
def unreserve(s): heappush(returned, s)
```

**Checklist:**
- [ ] Lấy ghế trả lại (heap) **trước** ghế mới (con trỏ)?
- [ ] Dùng **min-heap**, không dùng stack/queue?
- [ ] `unreserve` thật sự đưa ghế trở lại tập trống?
- [ ] Test sinh lệnh **hợp lệ** (đúng cam kết của đề)?
- [ ] Đã test `n = 1`, trả nhiều ghế theo thứ tự lộn xộn, `10⁵` lệnh?